# 39. Hetero PD and Serving Tiers | 异构 PD 与服务分层
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `Serving`, `Hetero PD`, `SLO` | **目标人群：** 推理服务学习者

---

## 本节导读

当 Prefill 与 Decode 不只位于不同逻辑池，而是运行在算力、显存、带宽或 backend 能力不同的资源上，分离本身不再足够。系统还要决定请求该路由到哪里、KV 状态该传输还是重算，以及资源紧张时如何保护服务目标。

本节把异构 PD 收成一条可检查的决策链：识别请求压力与资源画像，选择状态交接方式，再按照延迟和可运行性目标执行分层、回退或同池处理。

**关键词：** `heterogeneous PD`, `KV handoff`, `routing`, `SLO`, `fallback`

## 前置阅读

**导语：** 先理解 Chunked Prefill 和 PD 分池怎样改变请求的执行位置；再比较计算、显存与链路条件如何决定跨池交接、重算或保持同池。

- [38. Prefill/Decode Scheduling | Prefill/Decode 调度](./38_Prefill_Decode_Scheduling.ipynb)


### Step 1: 请求压力如何匹配异构资源

异构 PD 从请求阶段与资源能力的匹配开始：Prefill 更需要突发计算和输入带宽，Decode 更依赖持续 KV 容量、稳定读带宽与低排队。先将请求压力和资源画像写成同一张账本，才能判断哪种路由更合适。

| 对象 | 需要记录的特征 | 它影响的决策 |
|---|---|---|
| Prefill-heavy 请求 | prompt 长度、TTFT 目标、输入处理压力 | 是否进入计算更强的 prefill 池 |
| Decode-heavy 请求 | 生成长度、KV 驻留、TPOT 目标 | 是否进入显存/带宽更合适的 decode 池 |
| 资源池 | 计算能力、可用显存、链路和 backend 支持 | 能否承接对应阶段 |
| 服务等级 | P95/P99、超时与质量约束 | 是否允许排队、降级或回退 |

![异构 PD 总览：请求压力、资源画像、交接与服务动作](../docs/public/02_PyTorch_Algorithms/39_hetero_pd_overview.svg)

### Step 2: KV 状态交接应传输、重算还是同池执行

Prefill 完成后，Decode 需要已有的 KV 状态。跨池执行时，系统可传输状态、在目标池重算 Prefill，或继续留在原池；状态大小、链路带宽和资源空闲程度会改变三种选择的代价。

| 交接方式 | 适合条件 | 主要代价 | 必须验证 |
|---|---|---|---|
| KV 传输 | 链路足够快，状态传输低于重算预算 | 传输时间、通信拥塞、兼容性 | transfer bytes、handoff ms、P99 |
| 重算 Prefill | 链路慢或状态难以共享 | 额外 Prefill 计算与 TTFT | recompute ms、算力占用 |
| 保持同池 | 跨池交接代价高或资源未明显分化 | 失去隔离收益 | 队列、利用率、尾延迟 |

![Hetero PD 路由与状态交接](../docs/public/02_PyTorch_Algorithms/39_hetero_pd_route.svg)

### Step 3: 服务分层与回退如何保护 SLO

资源压力、链路退化或某个池持续饱和时，路由需要随当前风险改变。系统可继续异构路由、排队、限流、降级或回到同池；每个动作都要留下触发信号与结果，才能解释它保护了哪类请求。

| 触发信号 | 首选动作 | 保护目标 | 需要记录 |
|---|---|---|---|
| Decode 池显存接近上限 | 回到同池或限制新接纳 | 可运行性与已有请求 | memory pressure、拒绝数 |
| 交接超过预算 | 选择重算或同池执行 | P95/P99 | handoff、recompute、尾延迟 |
| 队列超出 SLO | 路由、限流或扩容 | 等待时间与公平性 | queue delay、超时率 |
| 两池均稳定 | 保持当前异构路由 | 吞吐与资源利用率 | pool utilization、throughput |

### Step 4: 实现资源路由、交接与服务动作

题目区先完成三项判断：请求需要哪类资源、KV 状态应传输还是重算、资源压力下应采取什么服务动作；路由记录由骨架统一生成，便于回看这三项选择。

| 实现阶段 | 机制责任 | 验证重点 |
|---|---|---|
| 请求分类 | 根据 prompt / decode 长度区分资源需要 | 阈值边界清楚 |
| 交接选择 | 比较传输时间与重算时间 | 慢链路不会错误选择传输 |
| 服务动作 | 根据显存、队列和 SLO 选择动作 | 风险优先级正确 |
| 路由记录 | 汇总阶段、交接和服务动作 | 决策理由可追踪 |

In [ ]:
from typing import Dict, List


In [ ]:
def classify_resource_need(request: Dict[str, int], long_prompt_threshold: int, long_decode_threshold: int) -> str:
    """根据 prompt 与 decode 长度返回 prefill、decode 或 shared 资源需求。"""
    # TODO 1（请求分类）：长 prompt 进入 prefill，长 decode 进入 decode，其余请求保持 shared。
    # resource_need = ???
    raise NotImplementedError('TODO 1：请完成资源需求分类')


def choose_handoff_mode(link_gbps: float, cache_transfer_mb: float, recompute_ms: float) -> Dict[str, object]:
    """比较 KV 传输与重算的教学成本，并选择耗时较低的路径。"""
    if link_gbps <= 0 or cache_transfer_mb < 0 or recompute_ms < 0:
        raise ValueError('链路、状态大小和重算时间必须合法')
    # TODO 2（状态交接）：计算 transfer_ms，并选择 transfer 或 recompute。
    # transfer_ms = ???  # cache_transfer_mb 转为 Gb 后除以 link_gbps，再换算为 ms。
    # mode = ???
    raise NotImplementedError('TODO 2：请完成交接方式选择')


def select_serving_action(memory_pressure: float, queue_delay_ms: float, slo_queue_budget_ms: float) -> Dict[str, str]:
    """在资源压力和队列等待之间选择路由、回退或限流动作。"""
    # TODO 3（服务动作）：先保护显存驻留，再检查 queue_delay 是否超过 SLO。
    # result = ???  # 返回 action 与 protect；正常情况下保持异构路由。
    raise NotImplementedError('TODO 3：请选择服务动作')


def build_route_record(request: Dict[str, int], handoff: Dict[str, object], action: Dict[str, str]) -> Dict[str, str]:
    """将三项机制判断整理为可复查的路由记录。"""
    return {
        'resource_need': str(request.get('resource_need', 'shared')),
        'handoff_mode': str(handoff['mode']),
        'service_action': action['action'],
        'protect': action['protect'],
    }


In [ ]:
# 机制测试：分别检查资源分类、handoff、服务动作和路由记录。
def test_resource_need_classification():
    """验证请求压力标签的阈值边界。"""
    assert classify_resource_need({'prompt_tokens': 3000, 'decode_tokens': 64}, 2048, 256) == 'prefill'
    assert classify_resource_need({'prompt_tokens': 128, 'decode_tokens': 512}, 2048, 256) == 'decode'
    assert classify_resource_need({'prompt_tokens': 800, 'decode_tokens': 128}, 2048, 256) == 'shared'


def test_handoff_mode_selection():
    """验证传输成本与重算成本的选择。"""
    fast_link = choose_handoff_mode(link_gbps=200, cache_transfer_mb=100, recompute_ms=12)
    assert fast_link == {'mode': 'transfer', 'transfer_ms': 4.0, 'recompute_ms': 12}
    slow_link = choose_handoff_mode(link_gbps=25, cache_transfer_mb=100, recompute_ms=12)
    assert slow_link['mode'] == 'recompute'


def test_serving_action_policy():
    """验证显存压力优先于队列延迟的服务动作。"""
    memory_action = select_serving_action(memory_pressure=0.92, queue_delay_ms=20, slo_queue_budget_ms=80)
    assert memory_action == {'action': 'fallback_to_shared', 'protect': 'residency'}
    latency_action = select_serving_action(memory_pressure=0.4, queue_delay_ms=120, slo_queue_budget_ms=80)
    assert latency_action == {'action': 'route_or_throttle', 'protect': 'latency'}


def test_route_record_contract():
    """验证路由记录保留资源、handoff 和保护目标。"""
    handoff = choose_handoff_mode(link_gbps=200, cache_transfer_mb=100, recompute_ms=12)
    action = select_serving_action(memory_pressure=0.4, queue_delay_ms=120, slo_queue_budget_ms=80)
    record = build_route_record({'resource_need': 'prefill'}, handoff, action)
    assert record == {'resource_need': 'prefill', 'handoff_mode': 'transfer', 'service_action': 'route_or_throttle', 'protect': 'latency'}


def test_heterogeneous_pd_policy():
    """汇总四组异构 PD 决策机制测试。"""
    try:
        test_resource_need_classification()
        test_handoff_mode_selection()
        test_serving_action_policy()
        test_route_record_contract()
        print('✅ 异构 PD 路由机制测试通过：资源分类、handoff、服务动作和路由记录均通过。')
    except NotImplementedError:
        raise
    except (NameError, AttributeError, TypeError, ValueError, AssertionError) as error:
        raise NotImplementedError('请先完成 TODO 代码或检查字段名！') from error


test_heterogeneous_pd_policy()


## 参考代码与解析

### 代码


In [ ]:
def classify_resource_need(request: Dict[str, int], long_prompt_threshold: int, long_decode_threshold: int) -> str:
    """根据 prompt 与 decode 长度返回 prefill、decode 或 shared 资源需求。"""
    # TODO 1：根据两类长度选择请求主要压力。
    prompt_tokens = request.get('prompt_tokens', 0)
    decode_tokens = request.get('decode_tokens', 0)
    if prompt_tokens > long_prompt_threshold and decode_tokens <= long_decode_threshold:
        return 'prefill'
    if decode_tokens > long_decode_threshold and prompt_tokens <= long_prompt_threshold:
        return 'decode'
    return 'shared'


def choose_handoff_mode(link_gbps: float, cache_transfer_mb: float, recompute_ms: float) -> Dict[str, object]:
    """比较 KV 传输与重算的教学成本，并选择耗时较低的路径。"""
    if link_gbps <= 0 or cache_transfer_mb < 0 or recompute_ms < 0:
        raise ValueError('链路、状态大小和重算时间必须合法')
    # TODO 2：将 MB 转为 Gb，估算 transfer_ms 后与 recompute_ms 比较。
    transfer_ms = (cache_transfer_mb * 8) / link_gbps
    mode = 'transfer' if transfer_ms <= recompute_ms else 'recompute'
    return {'mode': mode, 'transfer_ms': round(transfer_ms, 4), 'recompute_ms': recompute_ms}


def select_serving_action(memory_pressure: float, queue_delay_ms: float, slo_queue_budget_ms: float) -> Dict[str, str]:
    """在资源压力和队列等待之间选择路由、回退或限流动作。"""
    # TODO 3：显存压力优先于队列等待；两者正常时保持异构路由。
    if memory_pressure > 0.9:
        return {'action': 'fallback_to_shared', 'protect': 'residency'}
    if queue_delay_ms > slo_queue_budget_ms:
        return {'action': 'route_or_throttle', 'protect': 'latency'}
    return {'action': 'keep_heterogeneous_route', 'protect': 'throughput'}


def build_route_record(request: Dict[str, int], handoff: Dict[str, object], action: Dict[str, str]) -> Dict[str, str]:
    """将三项机制判断整理为可复查的路由记录。"""
    return {
        'resource_need': str(request.get('resource_need', 'shared')),
        'handoff_mode': str(handoff['mode']),
        'service_action': action['action'],
        'protect': action['protect'],
    }


### 解析

**TODO 1：分类资源压力**

- 长 prompt 且生成较短时更需要 Prefill 资源；长生成且 prompt 较短时更需要 Decode 资源；其余请求保留在 shared 路径。

**TODO 2：选择状态交接方式**

- `transfer_ms = cache_transfer_mb × 8 / link_gbps` 是把状态大小和链路带宽转成时间的教学估算。
- 传输比重算快时选择 `transfer`，否则选择 `recompute`；测试分别覆盖快链路与慢链路。

**TODO 3：选择服务动作**

- 显存压力先触发回到 shared 池，避免已有 Decode 请求失去驻留状态；队列超过 SLO 时再选择路由或限流。
- `build_route_record` 只是把前三项结果收成一条记录，方便在后续服务实验中核对路由原因。

### Step 5: 可选 GPU 机制探针：比较交接与重算成本

#### 5.1 环境、输入与固定条件

本实验用合成 KV 状态和矩阵计算近似比较交接与重算的成本。先固定状态大小、链路带宽、dtype 和重复次数，再记录传输字节、传输时间与重算时间。

| 对照项 | 必须固定或记录 | 用于判断 |
|---|---|---|
| 固定条件 | 状态大小、链路带宽、dtype、重复次数 | 让传输与重算在同一口径下比较 |
| 对照路径 | 合成交接与矩阵重算 | 比较 transfer ms 与 recompute ms |
| 记录字段 | handoff bytes、链路带宽、失败状态和 JSON | 解释成本来自何处 |

#### 5.2 配置与执行

执行单元默认关闭；它只生成机制探针 JSON。真实多资源 Serving 对照由 [70](./70_Serving_Scheduler_Benchmark.ipynb) 与 79–81 的多卡项目承接。

In [ ]:
"""机制探针配置：默认只记录配置，不启动模型或多 GPU backend。"""
RUN_GPU_HANDOFF_PROBE = False
GPU_RESULT_PATH = 'benchmarks/results/39_hetero_pd_handoff_probe.json'
GPU_DTYPE = 'float16'
GPU_REPEATS = 3
GPU_HIDDEN_SIZE = 512
HANDOFF_BYTES = 64 * 1024 * 1024
LINK_GBPS = 200.0
RECOMPUTE_MS = 8.0


In [ ]:
"""机制探针：比较合成 KV handoff 与重算成本，不伪造真实 PD 性能。"""
if RUN_GPU_HANDOFF_PROBE:
    import json
    import time
    from pathlib import Path
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('机制探针需要 CUDA；请先切换到 GPU runtime。')
    if GPU_DTYPE not in {'float16', 'bfloat16'}:
        raise ValueError('GPU_DTYPE 只能是 float16 或 bfloat16。')
    if GPU_DTYPE == 'bfloat16' and not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError('当前 GPU 不支持原生 BF16，请改用 float16。')

    dtype = getattr(torch, GPU_DTYPE)
    device = torch.device('cuda')
    weight = torch.randn(GPU_HIDDEN_SIZE, GPU_HIDDEN_SIZE, device=device, dtype=dtype)
    x = torch.randn(1, GPU_HIDDEN_SIZE, device=device, dtype=dtype)
    runs = []
    for _ in range(GPU_REPEATS):
        torch.cuda.synchronize()
        start = time.perf_counter()
        _ = x @ weight
        torch.cuda.synchronize()
        compute_ms = (time.perf_counter() - start) * 1000
        transfer_ms = HANDOFF_BYTES * 8 / (LINK_GBPS * 1e9) * 1000
        runs.append({'compute_ms': round(compute_ms, 4), 'transfer_ms': round(transfer_ms, 4), 'recompute_ms': RECOMPUTE_MS})

    project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'benchmarks').is_dir()), Path.cwd())
    report = {
        'task': 'hetero_pd_handoff_probe',
        'evidence_level': 'synthetic_gpu_probe',
        'config': {'dtype': GPU_DTYPE, 'hidden_size': GPU_HIDDEN_SIZE, 'handoff_bytes': HANDOFF_BYTES, 'link_gbps': LINK_GBPS, 'recompute_ms': RECOMPUTE_MS, 'repeats': GPU_REPEATS, 'device': torch.cuda.get_device_name(0)},
        'results': runs,
        'decision': {'decision': 'measure', 'reason': '仅比较合成 handoff 成本与重算参考，不代表真实 PD backend。'},
    }
    output_path = project_root / GPU_RESULT_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'GPU 机制探针结果已保存：{output_path}')
else:
    print('GPU 机制探针未启动：将 RUN_GPU_HANDOFF_PROBE 改为 True 后运行。')


In [ ]:
# 5.3 只读取已保存的机制探针 JSON；不会发起跨资源请求。
import json
from pathlib import Path

result_path = Path(GPU_RESULT_PATH)
if result_path.exists():
    report = json.loads(result_path.read_text(encoding='utf-8'))
    print({key: report.get(key) for key in ('task', 'config', 'results', 'evidence_level', 'decision')})
else:
    print(f'尚未找到结果文件：{result_path}。先在 5.2 运行机制探针。')


#### 5.4 记录结果与证据

每次复测新增一行，并区分 synthetic GPU probe 与真实多资源 backend。没有真实资源池、状态交接或链路数据时，相关字段保留为空，不用合成值代替服务结论。

#### GPU 机制探针记录

| workload | GPU / dtype | handoff bytes | link GB/s | transfer ms | recompute ms | evidence level | decision | JSON |
|---|---|---:|---:|---:|---:|---|---|---|
| synthetic handoff | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | synthetic_gpu_probe | measure | 待填写 |


## 相关阅读

完成资源画像、状态交接和服务动作判断后，可以继续阅读真实 PD 实现、多实例 Serving 与通信验证。

- [DistServe 原论文：Disaggregating Prefill and Decoding for Goodput-optimized Large Language Model Serving](https://arxiv.org/abs/2401.09670)
- [SGLang PD Disaggregation 文档](https://docs.sglang.ai/advanced_features/pd_disaggregation.html)
- [vLLM 官方仓库](https://github.com/vllm-project/vllm)
- [70. Serving Scheduler Benchmark | 服务调度基准](./70_Serving_Scheduler_Benchmark.ipynb)
- [79–81 分布式推理项目](./79_Distributed_Parallel_Benchmark.ipynb)